In [11]:
import json
import os
import pandas as pd
import re
import numpy as np

## **Data Loading and Data Pre Processing**

In [12]:
dataset_path = 'E:/JupyNote/DL Mini Projects/Contract Understanding Atticus Dataset (CAUD)/dataset'

In [13]:
dataset_files = os.listdir(dataset_path)
print(dataset_files)

['CUADv1.json', 'test.json', 'train_separate_questions.json']


In [14]:
with open(os.path.join(dataset_path, dataset_files[0]), 'r', encoding='utf-8') as f:
    caud_data = json.load(f)

In [15]:
print(len(caud_data))
print(caud_data.keys())

2
dict_keys(['version', 'data'])


In [16]:
contracts = []

for data in caud_data['data']:
    for paragraph in data['paragraphs']:
        for qa in paragraph['qas']:
            question = qa['question']

            match = re.search(r'"([^"]+)"', question)
            if not match:
                continue

            clause_type = match.group(1)

            for ans in qa['answers']:

                clause_text = ans.get('text', "")

                # If it's a list → take first element
                if isinstance(clause_text, list):
                    if len(clause_text) == 0:
                        continue
                    clause_text = clause_text[0]

                clause_text = clause_text.strip()

                if clause_text == "":
                    continue

                contracts.append({
                    'text': clause_text,
                    'label': clause_type
                })

In [17]:
print(len(contracts))

13822


In [32]:
contracts_df = pd.DataFrame(contracts)

In [33]:
contracts_df = contracts_df[contracts_df['text'].str.len() > 40]

In [34]:
contracts_df.head()

,text,label
7,The term of this Agreement shall be ten (10)...,Effective Date
8,Unless earlier terminated otherwise prov...,Effective Date
9,The term of this Agreement shall be ten (10)...,Expiration Date
10,If Distributor comp...,Renewal Term
11,This Agreement is to be construed according to...,Governing Law


In [35]:
contracts_df.shape

(10250, 2)

In [36]:
contracts_df['label'].nunique()

41

In [37]:
contracts_df['label'].value_counts().head()

label
License Grant       769
Cap On Liability    669
Anti-Assignment     642
Audit Rights        638
Insurance           552
Name: count, dtype: int64

## **Train test split**

In [38]:
from sklearn.model_selection import train_test_split

In [44]:
# First split: Train (80%) / Test (20%)
features_train, features_test, labels_train, labels_test = train_test_split(
    contracts_df['text'],
    contracts_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=contracts_df['label']
)

print(contracts_df['text'].shape, features_train.shape, labels_train.shape)
print(contracts_df['label'].shape, features_test.shape, labels_test.shape)

(10250,) (8200,) (8200,)
(10250,) (2050,) (2050,)


## **Label Encoding**

In [45]:
from sklearn.preprocessing import LabelEncoder
import pickle

In [46]:
encoder = LabelEncoder()

In [47]:
labels_train_encoding = encoder.fit_transform(labels_train)

In [48]:
labels_test_encoding = encoder.transform(labels_test)

In [49]:
encoder.classes_

array(['Affiliate License-Licensee', 'Affiliate License-Licensor',
       'Agreement Date', 'Anti-Assignment', 'Audit Rights',
       'Cap On Liability', 'Change Of Control',
       'Competitive Restriction Exception', 'Covenant Not To Sue',
       'Document Name', 'Effective Date', 'Exclusivity',
       'Expiration Date', 'Governing Law', 'Insurance',
       'Ip Ownership Assignment', 'Irrevocable Or Perpetual License',
       'Joint Ip Ownership', 'License Grant', 'Liquidated Damages',
       'Minimum Commitment', 'Most Favored Nation',
       'No-Solicit Of Customers', 'No-Solicit Of Employees',
       'Non-Compete', 'Non-Disparagement', 'Non-Transferable License',
       'Notice Period To Terminate Renewal', 'Parties',
       'Post-Termination Services', 'Price Restrictions', 'Renewal Term',
       'Revenue/Profit Sharing', 'Rofr/Rofo/Rofn', 'Source Code Escrow',
       'Termination For Convenience', 'Third Party Beneficiary',
       'Uncapped Liability', 'Unlimited/All-You-Can-Eat

In [50]:
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

## **Tokenization with Hugging Face**

In [62]:
import tensorflow as tf
tf.random.set_seed(3)
from tensorflow import keras
from transformers import AutoTokenizer, TFAutoModel

In [95]:
def tokenize_texts(texts, tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased"), max_len=64):
    return tokenizer(
        texts.tolist(),
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='tf'
    )
def tokenize_only_text(texts, tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased"), max_len=64):
    return tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='tf'
    )

E:\JupyNote\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [55]:
features_train_encoded = tokenize_texts(features_train)

In [63]:
distilbert_model = TFAutoModel.from_pretrained("distilbert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


In [64]:
max_len = 64

input_ids = tf.keras.layers.Input(
    shape=(max_len,),
    dtype=tf.int32,
    name="input_ids"
)

attention_mask = tf.keras.layers.Input(
    shape=(max_len,),
    dtype=tf.int32,
    name="attention_mask"
)

In [65]:
bert_output = distilbert_model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

In [66]:
cls_token = bert_output.last_hidden_state[:,0,:]

In [80]:
model = tf.keras.Model(
    inputs=[input_ids, attention_mask],
    outputs=tf.keras.layers.Dense(
        41,
        activation="sigmoid"
    )(
        tf.keras.layers.Dropout(0.3)(
            tf.keras.layers.Dense(
                64,
                activation="relu"
            )(
                tf.keras.layers.Dropout(0.3)(
                    tf.keras.layers.Dense(
                        128,
                        activation="relu"
                    )(cls_token)
                )
            )
        )
    )
)

In [81]:
model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 64)]                 0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 64)]                 0         []                            
 )                                                                                                
                                                                                                  
 tf_distil_bert_model (TFDi  TFBaseModelOutput(last_hid   6636288   ['input_ids[0][0]',           
 stilBertModel)              den_state=(None, 64, 768),   0          'attention_mask[0][0]']      
                              hidden_states=None, atten                                     

In [82]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [83]:
# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,                # Stop if no improvement for 5 epochs
    restore_best_weights=True
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_bert_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,                # Reduce LR by half
    patience=2,
    min_lr=1e-7
)

In [84]:
callback_list = [early_stopping, model_checkpoint, reduce_lr]

In [85]:
features_train_encoded.keys()
# 'input_ids' -> The main numerical representation of my text
# 'token_type_ids' -> Indicates which “sentence” a token belongs to.
# 'attention_mask' -> Indicates which tokens are real and which are padding. 1(Actual Token) 0 (Padding Token)

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [86]:
model.fit(
    {
        "input_ids": features_train_encoded["input_ids"],
        "attention_mask": features_train_encoded["attention_mask"]
    },
    labels_train_encoding,
    validation_split=0.1,
    epochs=30,
    batch_size=16,
    callbacks=callback_list
)

Epoch 1/30
462/462 [==============================] - ETA: 0s - loss: 2.8642 - accuracy: 0.2829

E:\JupyNote\.venv\Lib\site-packages\transformers\generation\tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


462/462 [==============================] - 524s 1s/step - loss: 2.8642 - accuracy: 0.2829 - val_loss: 2.3523 - val_accuracy: 0.4317 - lr: 3.0000e-05
Epoch 2/30
462/462 [==============================] - 556s 1s/step - loss: 2.4094 - accuracy: 0.4056 - val_loss: 2.1241 - val_accuracy: 0.4744 - lr: 3.0000e-05
Epoch 3/30
462/462 [==============================] - 520s 1s/step - loss: 2.1350 - accuracy: 0.4667 - val_loss: 1.8163 - val_accuracy: 0.5390 - lr: 3.0000e-05
Epoch 4/30
462/462 [==============================] - 524s 1s/step - loss: 1.8661 - accuracy: 0.5321 - val_loss: 1.7031 - val_accuracy: 0.5720 - lr: 3.0000e-05
Epoch 5/30
462/462 [==============================] - 521s 1s/step - loss: 1.5949 - accuracy: 0.6022 - val_loss: 1.6051 - val_accuracy: 0.5805 - lr: 3.0000e-05
Epoch 6/30
462/462 [==============================] - 513s 1s/step - loss: 1.3811 - accuracy: 0.6469 - val_loss: 1.5477 - val_accuracy: 0.6049 - lr: 3.0000e-05
Epoch 7/30
462/462 [==============================]

## **Checking accuracy on test data**

In [88]:
features_test_encoding = tokenize_texts(features_test)

In [91]:
loss, accuracy = model.evaluate(
    {
        "input_ids": features_test_encoding['input_ids'],
        "attention_mask": features_test_encoding['attention_mask']
    },
    labels_test_encoding
)

65/65 [==============================] - 31s 475ms/step - loss: 1.5974 - accuracy: 0.6039


In [94]:
print(f'Accuracy on testing data is {accuracy * 100 :.2f}%')

Accuracy on testing data is 60.39%


## **F1 Score**

In [144]:
testing_data_prediction = model.predict({
    "input_ids": features_test_encoding["input_ids"],
    "attention_mask": features_test_encoding["attention_mask"]
})

65/65 [==============================] - 31s 484ms/step


In [145]:
testing_data_prediction = testing_data_prediction.argmax(axis=1)

In [146]:
from sklearn.metrics import f1_score
f1 = f1_score(testing_data_prediction, labels_test_encoding, average='weighted')
print(f"F1 Score is {f1 * 100 :.2f}%")

F1 Score is 62.25%


## **Building a predictive system**

In [123]:
input_text = "Highlight the parts (if any) of this contract related to \"Revenue/Profit Sharing\" that should be reviewed by a lawyer. Details: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?"
input_text = input_text.strip()
input_text_encoding = tokenize_only_text(input_text)

In [124]:
input_data_prediction = model.predict(
    {
        "input_ids": input_text_encoding['input_ids'],
        "attention_mask": input_text_encoding['attention_mask']
    }
)

1/1 [==============================] - 0s 40ms/step


In [125]:
encoder.classes_[np.argmax(input_data_prediction[0])]

'Revenue/Profit Sharing'